<a href="https://colab.research.google.com/github/russellyaoduphd-cyber/520.637-HW-2/blob/main/AgentReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent reasoning：递归记忆、信息诊断与干预

**问题：**同样的交互历史经过不同记忆更新策略后，决定下一动作所需的信息是否仍然可用？

这是一个可复现的 **诊断实验原型**，不训练模型，也不是 MEM1 的复现或 AIS 充分性的证明。
不预设推理更新一定更好；两种策略都成功、都失败或没有差别，都是有效结果。

**先运行：**默认 `BACKEND="demo"`，点击 **Runtime → Run all**。这只检查实验管线，
规则程序不代表 LLM，所有表和导出文件会标明 `demo`。然后将配置改为 `BACKEND="hf"`，
在 Colab 选择 GPU 并重新从头运行，才会得到真实的开源模型结果。无需付费 API 或 API key。
首次会下载模型；默认 1.5B 仍是小模型，可把 `MODEL_ID` 改为更强的兼容模型。

Notebook 包含：配对任务、两种更新策略、完整历史基线、状态/历史检查器、
自然错误的最小修复（若存在）、单独标记的人工错误阳性对照、轨迹与计算量导出。


## 1. 符号与实验边界

沿用讨论中的含义：环境状态 $x_t$；观测 $o_t$；完整历史 $h_t$；
内部状态 $z_t\in\mathbb R^d$；决策 reasoning $r_t$；动作 $a_t$。
任务 $q$ 在所有 episode 中相同：两个开关相同选 RED，不同选 BLUE。奖励不用 $r_t$ 表示。

环境逐条发出线索。读取阶段的动作是 WAIT，$r_t$ 为空；
更新器 $F(z_t,r_t,a_t,o_{t+1})$ 将已有记忆和新观测压成新的 $z_{t+1}$。
全部线索结束后，决策器只读取最终 $z_t$，生成简短 $r_t$ 和最终动作 $a_t$。
**更新器内部生成的摘要是 $F$ 的计算，不与约定中的决策 $r_t$ 混用。**

本原型的 $z_t$ 是固定长度的 token-ID 数组，空位用 -1 填充；不是已学好的连续 latent，
也不把 token-ID 的欧氏距离解释为语义距离。固定任务规则属于所有策略共享的输入。
真实模型采用 greedy decoding，使更新规则在固定模型/运行设置下可重复。

两组只有更新指令不同：`facts` 保存开关事实，`reasoning` 尝试压缩成相同/不同的结论。
这检验 **提示词诱导的更新策略**，不能声称 facts 组的神经网络内部“没有推理”。
最终决策器、检查器模型、记忆上限和各阶段生成上限在两组间相同。


In [ ]:
# Configuration: edit this cell, then Run all.
BACKEND = "demo"                  # "demo" (plumbing only) or "hf" (real local LLM)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "main"            # Export records the resolved model commit when available.
SEED = 637
REPEATS_PER_SWITCH_PAIR = 1        # Each repeat contains all four (A, B) pairs.
DELAY_PAIRS = ((0, 0), (3, 3))     # (distractors between clues, distractors after both)
MEMORY_BUDGETS = (32,)             # Shared token cap; d is constant within each comparison.
UPDATE_MAX_NEW_TOKENS = 64
ACTOR_MAX_NEW_TOKENS = 32
CHECKER_MAX_NEW_TOKENS = 96        # Same stronger readout budget for z and z+h.
RUN_INTERVENTIONS = True
MAX_NATURAL_REPAIRS = 4            # Eligibility is defined before inspecting results.
BOOTSTRAP_SAMPLES = 2000
RUN_NAME = "switch_memory_v1"

assert BACKEND in {"demo", "hf"}
assert REPEATS_PER_SWITCH_PAIR >= 1
assert MEMORY_BUDGETS and all(isinstance(n, int) and n > 0 for n in MEMORY_BUDGETS)
assert DELAY_PAIRS and all(g >= 0 and t >= 0 for g, t in DELAY_PAIRS)


In [ ]:
# Only the real-LLM path installs model dependencies. No model download in demo mode.
import importlib.util
import subprocess
import sys

base_packages = [p for p in ("numpy", "pandas", "matplotlib")
                 if importlib.util.find_spec(p) is None]
if base_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *base_packages])
if BACKEND == "hf":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers==4.57.1", "accelerate>=1.0,<2", "torch>=2.2"])

import copy
import hashlib
import itertools
import json
import math
import platform
import random
import re
import time
import zipfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

random.seed(SEED)
np.random.seed(SEED)
TASK = ("Two switches A and B each have a fixed value, 0 or 1. "
        "If A and B have the SAME value, RED is safe. If they DIFFER, BLUE is safe. "
        "A stored relation=EQUAL is sufficient evidence for RED; relation=DIFFERENT "
        "is sufficient evidence for BLUE, even if the individual values were discarded. "
        "The switch values never change during an episode. "
        "Only observations explicitly reporting switch A or B are evidence. "
        "Room labels, display numbers and other scenery are irrelevant. "
        "At the final decision choose the safe door. A correct choice earns 1, otherwise 0.")
CONDITIONS = ("facts", "reasoning")


## 2. 可控环境与独立真值

每种延迟下均衡生成 `(A,B) = (0,0),(0,1),(1,0),(1,1)`，随机化线索顺序和干扰内容。
两种更新策略共享同一 episode。**真值只交给 evaluator，不传给更新器或决策器。**
完整历史是实际观测、WAIT 动作和空的读取阶段 reasoning，不包含隐藏真值。


In [ ]:
@dataclass(frozen=True)
class Episode:
    episode_id: str
    switch_a: int
    switch_b: int
    gap: int
    tail: int
    observations: tuple

    @property
    def correct_door(self):
        return "RED" if self.switch_a == self.switch_b else "BLUE"

    @property
    def correct_relation(self):
        return "EQUAL" if self.switch_a == self.switch_b else "DIFFERENT"

    def public_history(self):
        # This is the information actually seen by the agent, not hidden state.
        return [{"o_t": o, "r_t": "", "a_t": "WAIT"} for o in self.observations]


def make_episodes(seed, repeats, delay_pairs):
    rng = random.Random(seed)
    episodes = []
    serial = 0
    for gap, tail in delay_pairs:
        for repeat in range(repeats):
            pairs = list(itertools.product((0, 1), repeat=2))
            rng.shuffle(pairs)
            for a, b in pairs:
                clues = [f"Switch A = {a}.", f"Switch B = {b}."]
                rng.shuffle(clues)
                def distraction():
                    return (f"Scenery only: room label {rng.randrange(100, 1000)}; "
                            f"display digit {rng.randrange(2)}; wall {rng.choice(['plain', 'striped'])}.")
                observations = ([clues[0]] + [distraction() for _ in range(gap)]
                                + [clues[1]] + [distraction() for _ in range(tail)])
                episodes.append(Episode(f"ep-{serial:04d}", a, b, gap, tail, tuple(observations)))
                serial += 1
    return episodes


EPISODES = make_episodes(SEED, REPEATS_PER_SWITCH_PAIR, DELAY_PAIRS)
display(pd.DataFrame([{"episode": e.episode_id, "A (evaluator only)": e.switch_a,
                      "B (evaluator only)": e.switch_b, "gap": e.gap, "tail": e.tail,
                      "correct (evaluator only)": e.correct_door} for e in EPISODES]))


## 3. 两组提示词与模型后端

更新时只能读取旧 $z_t$ 和新观测，不偷偷回看 $h_t$。
facts 保留 `A=...; B=...`；reasoning 在证据齐全后可保留 `relation=EQUAL/DIFFERENT`。
这样的格式便于独立检查，但也偏向这个简单任务；不能泛化成所有 reasoning 的比较。

所有模型输出完整保留到日志。解析失败记为 `INVALID`，未知答案记为 `UNKNOWN`；
都计入失败，绝不从真值补答案。超过记忆容量时按同一规则截断，并统计截断率。


In [ ]:
UPDATE_COMMON = (
    "You maintain a bounded memory for a decision task. "
    "Read ONLY the supplied previous memory and new observation. "
    "Never invent an unseen switch value. Preserve relevant information on scenery-only turns. "
    "Return only the updated memory, with no preamble, markdown or explanation. "
)
UPDATE_INSTRUCTION = {
    "facts": ("Keep observed switch facts. Use A=0 or A=1, and B=0 or B=1, only when known. "
              "Use ? for an unknown value. Do not compute the relation or choose a door. "
              "Preferred format: A=?; B=?"),
    "reasoning": ("Reason about what will determine the safe door. When both switch values "
                  "are known, replace the facts by relation=EQUAL or relation=DIFFERENT. "
                  "If a previous relation is already known, retain it because switches never change. "
                  "Before both are known retain the available A=0/A=1 or B=0/B=1 fact. "
                  "Do not guess a relation before both values are known.")
}


def build_messages(kind, payload):
    if kind == "update":
        system = UPDATE_COMMON + UPDATE_INSTRUCTION[payload["condition"]]
        user = (TASK + f"\nMemory cap: {payload['budget']} tokenizer tokens.\n"
                + "Previous memory:\n" + (payload["memory"] or "(empty)")
                + "\nNew observation:\n" + payload["observation"])
    elif kind == "decide":
        system = ("Solve the supplied task using only the available evidence. "
                  "You may briefly check the facts. Do not invent missing information. "
                  "End with exactly one line FINAL=RED or FINAL=BLUE or FINAL=UNKNOWN. "
                  "A known relation is sufficient; do not demand the individual switch values. "
                  "Use FINAL=UNKNOWN only if neither both switch values nor a relation is known, "
                  "or if the available claims conflict.")
        user = TASK + "\nAvailable memory:\n" + (payload.get("memory", "") or "(none)")
        if "history" in payload:
            user += ("\nFull observed history (raw observations are authoritative if memory conflicts):\n"
                     + json.dumps(payload["history"], ensure_ascii=False))
    else:
        raise ValueError(kind)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def symbolic_decode(text):
    """Conservative decoder for the declared tiny schema, not a general semantic oracle."""
    claims = {k: {int(v) for v in re.findall(rf"\b{k}\s*=\s*([01])\b", text)}
              for k in ("A", "B")}
    relations = set(re.findall(r"\brelation\s*=\s*(EQUAL|DIFFERENT)\b", text, re.I))
    relations = {r.upper() for r in relations}
    recognized = bool(claims["A"] or claims["B"] or relations)
    if any(len(v) > 1 for v in claims.values()) or len(relations) > 1:
        return {"door": "UNKNOWN", "status": "conflicting_claims", "recognized": recognized}
    candidates = []
    for a, b in itertools.product((0, 1), repeat=2):
        if claims["A"] and a not in claims["A"]:
            continue
        if claims["B"] and b not in claims["B"]:
            continue
        relation = "EQUAL" if a == b else "DIFFERENT"
        if relations and relation not in relations:
            continue
        candidates.append("RED" if a == b else "BLUE")
    doors = set(candidates)
    status = "decodable" if len(doors) == 1 else ("conflicting_claims" if not doors else "inconclusive")
    return {"door": next(iter(doors)) if len(doors) == 1 else "UNKNOWN",
            "status": status, "recognized": recognized}


def parse_final(text):
    answers = re.findall(r"^\s*FINAL\s*=\s*(RED|BLUE|UNKNOWN)\s*[.!]?\s*$", text, re.M | re.I)
    return answers[0].upper() if len(answers) == 1 else "INVALID"


class DemoTokenizer:
    # One Unicode character per token. Do not compare demo token counts with HF counts.
    def encode(self, text, add_special_tokens=False):
        return [ord(c) for c in text]

    def decode(self, ids, **kwargs):
        return "".join(chr(int(i)) for i in ids)


class DemoBackend:
    name = "demo_symbolic_not_llm"
    tokenizer = DemoTokenizer()
    revision = "not-applicable"

    def generate(self, kind, payload, max_new_tokens):
        messages = build_messages(kind, payload)
        if kind == "update":
            text = payload["memory"] + "\n" + payload["observation"]
            values = {k: re.findall(rf"\b{k}\s*=\s*([01])\b", text) for k in ("A", "B")}
            facts = "; ".join(f"{k}={v[-1]}" for k, v in values.items() if v)
            prior = re.findall(r"\brelation=(EQUAL|DIFFERENT)\b", payload["memory"])
            if payload["condition"] == "reasoning" and values["A"] and values["B"]:
                out = "relation=" + ("EQUAL" if values["A"][-1] == values["B"][-1] else "DIFFERENT")
            elif payload["condition"] == "reasoning" and prior:
                out = "relation=" + prior[-1]
            else:
                out = facts or "A=?; B=?"
        else:
            evidence = payload.get("memory", "")
            if "history" in payload:
                evidence = "\n".join(step["o_t"] for step in payload["history"])
            out = "FINAL=" + symbolic_decode(evidence)["door"]
        out = out[:max_new_tokens]
        return {"text": out, "input_tokens": sum(len(m["content"]) for m in messages),
                "output_tokens": len(out), "seconds": 0.0, "hit_generation_cap": len(out) == max_new_tokens}


class HFBackend:
    def __init__(self, model_id, revision):
        import torch
        import transformers
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        self.name = model_id
        self.versions = {"torch": torch.__version__, "transformers": transformers.__version__}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        if self.device == "cpu":
            print("CPU inference selected; use a Colab GPU for a larger run.")
        torch.manual_seed(SEED)
        if self.device == "cuda":
            torch.cuda.manual_seed_all(SEED)
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, revision=revision, dtype=dtype, trust_remote_code=False).to(self.device).eval()
        self.revision = getattr(self.model.config, "_commit_hash", None) or revision

    def generate(self, kind, payload, max_new_tokens):
        messages = build_messages(kind, payload)
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(self.device)
        n_input = inputs["input_ids"].shape[-1]
        limit = getattr(self.model.config, "max_position_embeddings", 32768)
        if n_input + max_new_tokens > limit:
            raise ValueError("Prompt exceeds model context; no silent truncation of history is allowed.")
        if self.device == "cuda":
            self.torch.cuda.synchronize()
        start = time.perf_counter()
        with self.torch.inference_mode():
            output = self.model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens,
                                         pad_token_id=self.tokenizer.eos_token_id,
                                         use_cache=True)
        if self.device == "cuda":
            self.torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        ids = output[0, n_input:]
        return {"text": self.tokenizer.decode(ids, skip_special_tokens=True),
                "input_tokens": int(n_input), "output_tokens": int(ids.numel()),
                "seconds": elapsed, "hit_generation_cap": int(ids.numel()) == max_new_tokens}


backend = DemoBackend() if BACKEND == "demo" else HFBackend(MODEL_ID, MODEL_REVISION)
print("BACKEND:", backend.name)
if BACKEND == "demo":
    print("DEMO ONLY: tables below are software checks, not LLM findings.")


## 4. 固定维度状态与递归更新

只有这一段负责持久化记忆。完整历史保存在独立 evaluator 日志中；不传入 $F$。
更新原文、截断后的记忆、token 数和调用输入都会记录，便于核对遗忘发生在哪一轮。


In [ ]:
class MemoryCodec:
    def __init__(self, tokenizer, budget):
        if budget < 1:
            raise ValueError("Memory budget must be positive")
        self.tokenizer, self.budget = tokenizer, int(budget)

    def encode(self, text):
        ids = self.tokenizer.encode(text, add_special_tokens=False)
        z = np.full(self.budget, -1, dtype=np.int64)
        kept = ids[:self.budget]
        z[:len(kept)] = kept
        return z, len(ids) > self.budget

    def decode(self, z):
        if np.shape(z) != (self.budget,):
            raise ValueError("State has the wrong fixed dimension")
        return self.tokenizer.decode([int(i) for i in z if i >= 0], skip_special_tokens=True)

    def used(self, z):
        return int(np.sum(z >= 0))


CALL_LOG = []


def call_model(kind, payload, cap, context):
    # Only public payload is sent to the backend; context contains log IDs, not truth.
    result = backend.generate(kind, payload, cap)
    entry = {"call_id": len(CALL_LOG), "backend": backend.name, "kind": kind,
             "cap": cap, **context, "payload": copy.deepcopy(payload), **result}
    CALL_LOG.append(entry)
    return entry


def F(z_t, r_t, a_t, o_next, *, codec, condition, context):
    """Reading-phase state transition; the previous public action is WAIT."""
    if a_t != "WAIT" or r_t != "":
        raise ValueError("This toy environment updates memory only during WAIT steps.")
    payload = {"condition": condition, "budget": codec.budget,
               "memory": codec.decode(z_t), "observation": o_next}
    result = call_model("update", payload, UPDATE_MAX_NEW_TOKENS, context)
    z_next, truncated = codec.encode(result["text"])
    return z_next, {"update_call_id": result["call_id"], "raw_update": result["text"],
                    "memory": codec.decode(z_next), "memory_tokens": codec.used(z_next),
                    "memory_truncated": truncated}


def decide(memory, *, cap, context, history=None):
    payload = {"memory": memory}
    if history is not None:
        payload["history"] = history
    result = call_model("decide", payload, cap, context)
    return {"action": parse_final(result["text"]), "r_t": result["text"],
            "call_id": result["call_id"]}


## 5. 配对实验

每个 episode 先跑完整历史 actor 基线，然后对两种更新策略分别运行：

1. `actor_z`：最终 actor 只读压缩状态。
2. `checker_z`：同一模型、较大的生成上限，只读状态。
3. `checker_zh`：同一个 checker、相同上限，读状态 + 完整历史。
4. `schema_decodable`：独立的精确小型 schema decoder 是否能从显式事实/关系确定正确门。

`checker_z` 失败而 `checker_zh` 成功，是历史中可能仍有有用信息的证据；不是遗忘的证明。
schema decoder 不理解自由文本，它的失败也可能只是格式不匹配。二者均不等同于 Markov 性。


In [ ]:
RESULTS, TRAJECTORIES, BASELINES = [], [], []
for ep_index, episode in enumerate(EPISODES):
    history = episode.public_history()
    base = decide("", cap=ACTOR_MAX_NEW_TOKENS,
                  context={"episode_id": episode.episode_id, "stage": "full_history_actor"},
                  history=history)
    BASELINES.append({"episode_id": episode.episode_id, "gap": episode.gap, "tail": episode.tail,
                      "action": base["action"], "correct": base["action"] == episode.correct_door})
    for budget in MEMORY_BUDGETS:
        codec = MemoryCodec(backend.tokenizer, budget)
        # Alternate order to reduce systematic runtime/order effects.
        order = CONDITIONS if ep_index % 2 == 0 else tuple(reversed(CONDITIONS))
        for condition in order:
            context = {"episode_id": episode.episode_id, "condition": condition, "budget": budget}
            z, _ = codec.encode("")
            trajectory = []
            for t, observation in enumerate(episode.observations):
                prior = codec.decode(z)
                z, info = F(z, "", "WAIT", observation, codec=codec, condition=condition,
                            context={**context, "stage": "update", "step": t + 1})
                trajectory.append({**context, "step": t + 1, "observation": observation,
                                   "prior_memory": prior, "z_t": z.tolist(), **info})
            TRAJECTORIES.extend(trajectory)
            memory = codec.decode(z)
            actor = decide(memory, cap=ACTOR_MAX_NEW_TOKENS, context={**context, "stage": "actor_z"})
            check_z = decide(memory, cap=CHECKER_MAX_NEW_TOKENS, context={**context, "stage": "checker_z"})
            check_zh = decide(memory, cap=CHECKER_MAX_NEW_TOKENS,
                              context={**context, "stage": "checker_zh"}, history=history)
            symbolic = symbolic_decode(memory)
            RESULTS.append({**context, "backend": backend.name, "gap": episode.gap, "tail": episode.tail,
                            "truth": episode.correct_door, "final_memory": memory, "z_t": z.tolist(),
                            "memory_tokens": codec.used(z),
                            "truncated_updates": sum(x["memory_truncated"] for x in trajectory),
                            "num_updates": len(trajectory),
                            "full_history_actor_correct": base["action"] == episode.correct_door,
                            "actor_z": actor["action"], "actor_r_t": actor["r_t"],
                            "checker_z": check_z["action"], "checker_zh": check_zh["action"],
                            "actor_correct": actor["action"] == episode.correct_door,
                            "checker_z_correct": check_z["action"] == episode.correct_door,
                            "checker_zh_correct": check_zh["action"] == episode.correct_door,
                            "schema_status": symbolic["status"],
                            "schema_decodable": symbolic["door"] == episode.correct_door,
                            "schema_recognized": symbolic["recognized"]})
    print(f"Completed {ep_index + 1}/{len(EPISODES)} episodes ({backend.name})")

results = pd.DataFrame(RESULTS)
trajectories = pd.DataFrame(TRAJECTORIES)
baselines = pd.DataFrame(BASELINES)
display(results[["episode_id", "condition", "budget", "gap", "tail", "final_memory",
                 "truth", "actor_z", "checker_z", "checker_zh", "schema_status"]])


## 6. 结果、配对差异与计算量

准确率包括 UNKNOWN/INVALID，不能只统计成功解析的输出。
误差条是 Wilson 95% 区间；差异用 **同 episode 配对 bootstrap**，每个延迟层内重采样。
这只是描述性不确定性：固定小环境、固定 greedy 模型和很小的 smoke 样本不能支撑广泛结论。
生成预算上限相同不代表实际计算量相同，所以一并报告实际 input/output tokens 与耗时。


In [ ]:
def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return center - radius, center + radius


summary_rows = []
for (budget, condition), group in results.groupby(["budget", "condition"]):
    low, high = wilson_interval(int(group.actor_correct.sum()), len(group))
    summary_rows.append({"backend": backend.name, "budget": budget, "condition": condition,
                         "n": len(group), "actor_accuracy": group.actor_correct.mean(),
                         "ci_low": low, "ci_high": high,
                         "full_history_actor_accuracy": group.full_history_actor_correct.mean(),
                         "checker_z_accuracy": group.checker_z_correct.mean(),
                         "checker_zh_accuracy": group.checker_zh_correct.mean(),
                         "history_rescue_rate": ((~group.checker_z_correct) & group.checker_zh_correct).mean(),
                         "schema_decodable_rate": group.schema_decodable.mean(),
                         "actor_invalid_rate": (group.actor_z == "INVALID").mean(),
                         "actor_unknown_rate": (group.actor_z == "UNKNOWN").mean(),
                         "update_truncation_rate": group.truncated_updates.sum() / group.num_updates.sum(),
                         "mean_memory_tokens": group.memory_tokens.mean()})
summary = pd.DataFrame(summary_rows)
display(summary)
display(results.groupby(["budget", "condition", "gap", "tail"])
        [["actor_correct", "checker_z_correct", "checker_zh_correct", "schema_decodable"]].mean())

paired_rows = []
rng = np.random.default_rng(SEED)
for budget in MEMORY_BUDGETS:
    data = results[results.budget == budget]
    strata = []
    for (gap, tail), group in data.groupby(["gap", "tail"]):
        wide = group.pivot(index="episode_id", columns="condition", values="actor_correct").astype(float)
        strata.append((wide["reasoning"] - wide["facts"]).to_numpy())
    observed = np.concatenate(strata).mean()
    bootstrap = np.array([np.concatenate([rng.choice(d, len(d), replace=True) for d in strata]).mean()
                          for _ in range(BOOTSTRAP_SAMPLES)])
    paired_rows.append({"budget": budget, "reasoning_minus_facts": observed,
                        "paired_ci_low": np.quantile(bootstrap, .025),
                        "paired_ci_high": np.quantile(bootstrap, .975)})
paired_differences = pd.DataFrame(paired_rows)
display(paired_differences)

calls = pd.DataFrame(CALL_LOG)
costs = calls[calls.stage.isin(["update", "actor_z", "checker_z", "checker_zh"])]
display(costs.groupby(["budget", "condition", "stage"])
        [["input_tokens", "output_tokens", "seconds", "hit_generation_cap"]].agg(["sum", "mean"]))

fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(summary))
ax.bar(x, summary.actor_accuracy, color=["#4C78A8" if c == "facts" else "#F58518" for c in summary.condition])
ax.errorbar(x, summary.actor_accuracy,
            yerr=[np.maximum(0, summary.actor_accuracy-summary.ci_low),
                  np.maximum(0, summary.ci_high-summary.actor_accuracy)], fmt="none", color="black", capsize=4)
ax.set_xticks(x, [f"{r.condition}\ncap={r.budget}" for r in summary.itertuples()])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Final decision accuracy")
ax.set_title("DEMO: software check, not LLM evidence" if BACKEND == "demo" else f"Paired switch task | {MODEL_ID}")
fig.tight_layout()
plt.show()


## 7. 干预：自然错误与人工阳性对照分开

**自然错误：**只选择实际选错且记忆中存在唯一错误 `relation=...` 的样本，
只替换这个关系值，再用同一 actor 决策；另作原记忆不变的 sham 重放。
没有符合条件的错误，就报告没有，不能人为制造“自然失败”。

**人工阳性对照：**故意构造相反的关系值，然后修复。它只测试决策器是否对这条记忆敏感。
它使用 evaluator 真值，是诊断用 oracle 干预，不是 agent 的部署方法，不计入主准确率。
关系字符串 token 长度可能不同，记录实际 token 数；因此不宣称严格的等计算量因果效应。


In [ ]:
INTERVENTIONS = []
episodes_by_id = {e.episode_id: e for e in EPISODES}


def intervention_case(memory, repaired, *, source, episode, condition, budget):
    codec = MemoryCodec(backend.tokenizer, budget)
    z_bad, bad_truncated = codec.encode(memory)
    z_fixed, fixed_truncated = codec.encode(repaired)
    bad, fixed = codec.decode(z_bad), codec.decode(z_fixed)
    if bad_truncated or fixed_truncated:
        return {"source": source, "episode_id": episode.episode_id, "condition": condition,
                "budget": budget, "status": "skipped: intervention exceeds memory cap"}
    context = {"episode_id": episode.episode_id, "condition": condition, "budget": budget,
               "source": source}
    before = decide(bad, cap=ACTOR_MAX_NEW_TOKENS, context={**context, "stage": "intervention_before"})
    sham = decide(bad, cap=ACTOR_MAX_NEW_TOKENS, context={**context, "stage": "intervention_sham"})
    after = decide(fixed, cap=ACTOR_MAX_NEW_TOKENS, context={**context, "stage": "intervention_after"})
    return {**context, "status": "evaluated", "memory_before": bad, "memory_after": fixed,
            "tokens_before": codec.used(z_bad), "tokens_after": codec.used(z_fixed),
            "truth": episode.correct_door, "before": before["action"], "sham": sham["action"],
            "after": after["action"], "before_correct": before["action"] == episode.correct_door,
            "after_correct": after["action"] == episode.correct_door,
            "action_changed": before["action"] != after["action"]}


if RUN_INTERVENTIONS:
    n_natural = 0
    for row in RESULTS:
        if n_natural >= MAX_NATURAL_REPAIRS or row["actor_correct"]:
            continue
        episode = episodes_by_id[row["episode_id"]]
        matches = re.findall(r"\brelation\s*=\s*(EQUAL|DIFFERENT)\b", row["final_memory"], re.I)
        if len(matches) != 1 or matches[0].upper() == episode.correct_relation:
            continue
        repaired = re.sub(r"(\brelation\s*=\s*)(EQUAL|DIFFERENT)\b",
                          lambda m: m.group(1) + episode.correct_relation,
                          row["final_memory"], count=1, flags=re.I)
        INTERVENTIONS.append(intervention_case(row["final_memory"], repaired, source="natural_error",
                              episode=episode, condition=row["condition"], budget=row["budget"]))
        n_natural += 1
    print("Eligible natural errors examined:", n_natural)
    # One equal and one different episode per budget; never counted as natural failures.
    for budget in MEMORY_BUDGETS:
        for relation in ("EQUAL", "DIFFERENT"):
            episode = next(e for e in EPISODES if e.correct_relation == relation)
            wrong = "DIFFERENT" if relation == "EQUAL" else "EQUAL"
            INTERVENTIONS.append(intervention_case("relation=" + wrong, "relation=" + relation,
                                  source="synthetic_positive_control", episode=episode,
                                  condition="diagnostic_only", budget=budget))
interventions = pd.DataFrame(INTERVENTIONS)
if len(interventions):
    display(interventions)
else:
    print("No interventions run.")


## 8. 打开一条轨迹看状态在哪里改变

默认显示一个自然失败；如果没有则显示第一条。可以修改 `selected_id/selected_condition`。
状态中的事实、关系与最终动作分开显示，方便讨论“信息丢失”与“读出失败”。


In [ ]:
failures = results[~results.actor_correct]
selected = failures.iloc[0] if len(failures) else results.iloc[0]
selected_id = selected.episode_id
selected_condition = selected.condition
selected_budget = selected.budget
display(trajectories[(trajectories.episode_id == selected_id)
                     & (trajectories.condition == selected_condition)
                     & (trajectories.budget == selected_budget)]
        [["step", "observation", "prior_memory", "raw_update", "memory", "memory_truncated"]])
print("Final actor reasoning/output:", selected.actor_r_t)
print("Oracle answer (evaluation only):", selected.truth)


## 9. 实现检查

下面验证：所有真值组合、固定状态容量、截断、未知/冲突处理、同一公开历史配对、
以及更新器确实没有拿到完整历史或隐藏答案。demo 的无截断主实验应该两组都成功，
这是预期的管线检查，不人为注入“reasoning 更好”的结果。


In [ ]:
def run_checks():
    for a, b in itertools.product((0, 1), repeat=2):
        truth = "RED" if a == b else "BLUE"
        assert symbolic_decode(f"A={a}; B={b}")["door"] == truth
    assert symbolic_decode("A=0")["door"] == "UNKNOWN"
    assert symbolic_decode("A=0; A=1")["status"] == "conflicting_claims"
    assert symbolic_decode("A=0; B=0; relation=DIFFERENT")["status"] == "conflicting_claims"
    assert parse_final("Brief check.\nFINAL=BLUE") == "BLUE"
    assert parse_final("RED or BLUE") == "INVALID"
    assert parse_final("FINAL=RED\nFINAL=BLUE") == "INVALID"
    tiny = MemoryCodec(DemoTokenizer(), 3)
    encoded, cut = tiny.encode("abcdef")
    assert cut and encoded.shape == (3,) and tiny.decode(encoded) == "abc"
    empty, cut = tiny.encode("")
    assert not cut and tiny.decode(empty) == "" and np.all(empty == -1)
    assert make_episodes(SEED, REPEATS_PER_SWITCH_PAIR, DELAY_PAIRS) == EPISODES
    for gap, tail in DELAY_PAIRS:
        selected_eps = [e for e in EPISODES if e.gap == gap and e.tail == tail]
        assert {e.correct_door for e in selected_eps} == {"RED", "BLUE"}
    for call in CALL_LOG:
        if call["kind"] == "update":
            assert set(call["payload"]) == {"condition", "budget", "memory", "observation"}
        if call.get("stage") in ("actor_z", "checker_z"):
            assert set(call["payload"]) == {"memory"}
    for row in RESULTS:
        assert len(row["z_t"]) == row["budget"]
        assert row["memory_tokens"] <= row["budget"]
    for (episode_id, budget), group in trajectories.groupby(["episode_id", "budget"]):
        left = group[group.condition == "facts"].sort_values("step").observation.tolist()
        right = group[group.condition == "reasoning"].sort_values("step").observation.tolist()
        assert left == right
    # Explicit decoder evidence can rescue a deliberately false relation using public observations.
    demo = DemoBackend()
    history = [{"o_t": "Switch A = 0."}, {"o_t": "Switch B = 0."}]
    assert parse_final(demo.generate("decide", {"memory": "relation=DIFFERENT"}, 32)["text"]) == "BLUE"
    assert parse_final(demo.generate("decide", {"memory": "relation=DIFFERENT", "history": history}, 32)["text"]) == "RED"
    if BACKEND == "demo" and min(MEMORY_BUDGETS) >= 18 and UPDATE_MAX_NEW_TOKENS >= 18:
        assert results.actor_correct.all() and results.checker_z_correct.all()
    print("PASS: environment, parser, fixed memory, paired observations and input isolation checks.")


run_checks()


## 10. 导出与解释

导出 summary/results/trajectories/interventions、全部调用输入输出和运行配置。
`z_t` 为定长 token ID；tokenizer/model revision 写入 manifest。所有任务都是合成任务，无真实个人信息。
文件保存在本次 Colab runtime，可用最后一行（取消注释）下载 ZIP。

解释顺序：先看完整历史基线是否会做题，再看压缩 actor、checker 的历史增益，
最后看实际错误的轨迹和自然干预。若完整历史也失败，先处理模型能力/提示词问题。
若两组都满分，本任务不足以区分；增加干扰或更换更复杂任务，不应人为给某组制造错误。

**边界：**这是冻结模型的提示词实验，不是 RL 训练；没有验证跨所有历史/动作的 AIS 条件，
也没有证明 Markov 性或新的理论贡献。检查器有能力限制，prompt 形式也可能影响结果。


In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = Path("agent_reasoning_runs") / f"{RUN_NAME}_{BACKEND}_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=False)
for name, frame in {"results": results, "summary": summary, "paired_differences": paired_differences,
                    "trajectories": trajectories, "baselines": baselines,
                    "interventions": interventions}.items():
    frame.to_csv(output_dir / f"{name}.csv", index=False)
with (output_dir / "calls.jsonl").open("w", encoding="utf-8") as handle:
    for entry in CALL_LOG:
        handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
with (output_dir / "trajectories.jsonl").open("w", encoding="utf-8") as handle:
    for entry in TRAJECTORIES:
        handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
manifest = {
    "experiment_version": "1.0", "backend": BACKEND, "model": backend.name,
    "resolved_revision": backend.revision, "seed": SEED, "python": platform.python_version(),
    "versions": getattr(backend, "versions", {}), "memory_budgets": list(MEMORY_BUDGETS),
    "delay_pairs": list(DELAY_PAIRS), "repeats": REPEATS_PER_SWITCH_PAIR,
    "update_cap": UPDATE_MAX_NEW_TOKENS, "actor_cap": ACTOR_MAX_NEW_TOKENS,
    "checker_cap": CHECKER_MAX_NEW_TOKENS, "decoding": "greedy",
    "tokenization": "Unicode characters" if BACKEND == "demo" else "model tokenizer",
    "task": TASK, "update_common": UPDATE_COMMON, "update_instructions": UPDATE_INSTRUCTION,
    "episodes": [asdict(e) for e in EPISODES], "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "run_interventions": RUN_INTERVENTIONS, "max_natural_repairs": MAX_NATURAL_REPAIRS,
    "warning": "Synthetic software check, NOT LLM evidence" if BACKEND == "demo" else "Small diagnostic, not an AIS/Markov proof",
}
manifest["data_sha256"] = hashlib.sha256(json.dumps(manifest["episodes"], sort_keys=True).encode()).hexdigest()
(output_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
fig.savefig(output_dir / "accuracy.png", dpi=160, bbox_inches="tight")
zip_path = output_dir.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(output_dir.iterdir()):
        archive.write(path, arcname=path.name)
print("Saved:", zip_path)
# In Colab, uncomment these two lines to download:
# from google.colab import files
# files.download(str(zip_path))


## References

- Subramanian et al., [Approximate Information State for Approximate Planning and Reinforcement Learning in Partially Observed Systems](https://jmlr.org/papers/v23/20-1165.html), JMLR 2022. Definitions 3/7 and Theorem 9 motivate the diagnostic questions; this notebook does not establish their hypotheses.
- Zhou et al., [MEM1](https://arxiv.org/abs/2506.15841), 2025. Inspiration for recursive memory; no claim of reproducing its RL training or benchmarks.
- [Qwen2.5-1.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).
- [Transformers chat templates](https://huggingface.co/docs/transformers/chat_templating) and [generation](https://huggingface.co/docs/transformers/main_classes/text_generation).
